In [1]:
import ipdb

In [2]:
import header
from header import __root__

In [3]:
from src.webdrivers.pydoll import Driver

🔑 Found password in password.txt (DEBUG MODE)
✅ Successfully opened KeePass database: C:\Users\user\Documents\repos\hypotez\secrets\credentials.kdbx
Failed to extract Aliexpress API key from KeePass 'types.SimpleNamespace' object has no attribute 'aliexpress_com'
Failed to load Aliexpress credentials
Failed to load GAPI credentials
Unexpected exception formatting exception. Falling back to standard exception


Traceback (most recent call last):
  File "C:\Users\user\Documents\repos\hypotez\venv\Lib\site-packages\IPython\core\interactiveshell.py", line 3577, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
    ~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\user\AppData\Local\Temp\ipykernel_16144\1892718539.py", line 1, in <module>
    from src.webdrivers.pydoll import Driver
ModuleNotFoundError: No module named 'src.webdrivers'

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "C:\Users\user\Documents\repos\hypotez\venv\Lib\site-packages\IPython\core\interactiveshell.py", line 2168, in showtraceback
    stb = self.InteractiveTB.structured_traceback(
        etype, value, tb, tb_offset=tb_offset
    )
  File "C:\Users\user\Documents\repos\hypotez\venv\Lib\site-packages\IPython\core\ultratb.py", line 1454, in structured_traceback
    return FormattedTB.structured_traceback(
           ~~~~~~~~~~~~

In [ ]:

from src.suppliers.get_graber_by_supplier import get_graber_by_supplier_url

In [ ]:

async def run_scenario_async(
        self,
        mexiron_name:str,
        urls: List[str],
        price: str = "",
        bot: Optional[telebot.TeleBot] = None,
        chat_id: int = 0,
        attempts: int = 3,
    ) -> bool:
        """Запускает сценарий.

        Args:
            urls: Ссылки на товары (или категории).
            price: Цена для отчёта.
            bot: Телеграм‑бот для отправки статуса.
            chat_id: Идентификатор чата.
            attempts: Количество попыток перезапуска драйвера.

        Returns:
            bool: ``True`` при успешном завершении сценария.
        """
        products_list: list[dict] = []  # Список собранных товаров
        required_fields: list[str] = [
            "id_supplier",
            "name",
            "price",
            "reference",
            "description",
            "description_short",
            "specification",
            "default_image_url",
        ]
        try:
            driver = Driver(
                window_mode = Config.WINDOW_MODE,
                user_data_dir = Config.user_data_dir,
            )
        except Exception as ex:
            logger.error("Ошибка создания Driver", ex, exc_info=True)
            raise RuntimeError("Driver initialization failed") from ex

        # Парсинг страниц товаров ------------------------------------------------
        async with driver:

            # Запуск браузера ------------------------------------------------------
            try:
                await driver.start()
                # await driver.async_init_page() # <- В случае использования конекстного менеджера определяется в методе `__aenter__` драйвера 
            except Exception as ex:  
                logger.error("❌ Ошибка запуска pydoll драйвера", ex, exc_info=True)
                return False

            # Сбор товаров ---------------------------------------------------------
            for url in urls:
                logger.debug(f"Обработка URL: {url}", None, False)

                graber = get_graber_by_supplier_url(url, driver)

                if not graber:
                    logger.error(f"🤷‍♂️ Нет подходящего грабера для URL: {url}", None, True)
                    ipdb.set_trace()
                    continue
        
                try:
                    await driver.get_url(url)
                    product_fields: ProductFields = await graber.grab_page_async(required_fields = required_fields)
                except Exception as ex: 
                    logger.error(f"❌ Ошибка парсинга страницы:{url}", ex, exc_info = True)
                    if bot:
                        bot.send_message(chat_id, f"❌ Ошибка парсинга страницы:\n{url}\n{ex}")
                    continue

                if not product_fields or not product_fields.name:
                    logger.error(f"""❌ Ошибка парсинга товара:{url}
                    Проверьте локаторы.""", None, False, text_color="light_gray", bg_color="light_gray")
                    continue

                return 
                try:
                    # Конвертиртация поля из объекта `ProductFields` в простой словарь для модели llm
                    product_data = self.convert_product_fields(product_fields)

                    # Индивидуальные настройки поставщиков
                    match(graber.supplier_prefix):
                        case 'morlevi.co.il':
                            product_data['default_image_url'] = fr'https"://"morlevi.co.il/' + product_data['default_image_url'] 
                            ...
                        case 'grandadvance.co.il':
                            ...
                        case 'ksp.co.il':
                            ...
                        case 'ivory.co.il':
                            ...

                except Exception as ex:  
                    logger.error("Ошибка конвертации данных", ex, exc_info=True)
                    if bot:
                        bot.send_message(chat_id, f"❌ Ошибка конвертации:\n{url}")
                    continue

                products_list.append(product_data)
            return products_list


In [ ]:
def run_sample_scenario() -> None:
    """Пример локального теста сценария."""
    urls_list: list[str] = [
        "https://www.morlevi.co.il/product/21039",
        "https://www.morlevi.co.il/product/21018",
        "https://www.ivory.co.il/catalog.php?id=85473",
        "https://grandadvance.co.il/eng/?go=products&action=view&ties_ids=801&product_id=28457--SAMSUNG-SSD-1TB-990-EVO-PCle-4.0-x4--5.0-x2-NVMe",
    ]

    scenario = Scenario()
    logger.info("Запуск тестового сценария…")

    async def _runner() -> bool:
        if await scenario.run_scenario_async(
            urls=urls_list,
            mexiron_name="test_kazarinov_run",
            price="100.50",
            # bot=your_telebot_instance,
            # chat_id=your_chat_id,
            ):
            logger.info("Тестовый сценарий завершён")
            return True
        return

    asyncio.run(_runner())

In [ ]:
run_sample_scenario()